# Qur'an Recitation Fine-Tuning with NeMo
## NVIDIA Hybrid Conformer + Forced Alignment on Colab

**This notebook will:**
1. Install NeMo and dependencies
2. Download your dataset from public URL
3. Fine-tune the `nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0` model
4. Generate forced alignments
5. Download results back to your machine

**Estimated runtime: 2-3 hours on Colab T4 GPU**

In [ ]:
import os
import sys

# Check GPU
!nvidia-smi

print("\n=== Installing NeMo ===")
!pip install -q nemo_toolkit[asr]
print("\n✅ NeMo installed")

In [ ]:
# USER CONFIGURATION
DATASET_URL = "https://drive.google.com/uc?export=download&id=1sOjt2Svjk0pIM52uVwKs0h5imQanOnuh"

# Training config
NUM_EPOCHS = 3
LEARNING_RATE = 1e-4
BATCH_SIZE = 16  # Reduce to 8 if OOM errors
ACCUMULATE_GRAD_BATCHES = 2

# Paths
WORK_DIR = "/content/quran_finetune"
DATASET_DIR = f"{WORK_DIR}/dataset"
CHECKPOINT_DIR = f"{WORK_DIR}/checkpoints"
ALIGNMENTS_DIR = f"{WORK_DIR}/alignments"

# Model
PRETRAINED_MODEL = "nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0"

# Create directories
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(ALIGNMENTS_DIR, exist_ok=True)

print(f"✅ Config ready:")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   LR: {LEARNING_RATE}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Work Dir: {WORK_DIR}")

## Download Dataset

Downloads from your Google Drive link and extracts to working directory.

In [ ]:
import subprocess
import tarfile

os.chdir(WORK_DIR)

print("📥 Downloading dataset...")
!wget -q {DATASET_URL} -O dataset.tar.gz

print("📦 Extracting dataset...")
with tarfile.open("dataset.tar.gz", "r:gz") as tar:
    tar.extractall()

!mv data {DATASET_DIR}
!rm dataset.tar.gz

print(f"✅ Dataset extracted to {DATASET_DIR}")
print("\n📊 Dataset structure:")
!ls -lh {DATASET_DIR}

In [ ]:
import json

# Check manifests exist
for split in ['train', 'test', 'dev']:
    manifest = f"{DATASET_DIR}/manifest_{split}.json"
    if os.path.exists(manifest):
        with open(manifest) as f:
            count = sum(1 for _ in f)
        print(f"✅ manifest_{split}.json: {count} samples")
    else:
        print(f"❌ manifest_{split}.json NOT FOUND")

# Sample manifest entry
print("\n📋 Sample manifest entry:")
with open(f"{DATASET_DIR}/manifest_train.json") as f:
    sample = json.loads(f.readline())
    print(json.dumps(sample, indent=2))

## Load and Fine-Tune Model

In [ ]:
from nemo.collections.asr import models
from omegaconf import OmegaConf

print(f"📥 Loading pretrained model: {PRETRAINED_MODEL}")
model = models.EncDecRNNTBPEModel.from_pretrained(PRETRAINED_MODEL)

print("\n✅ Model loaded successfully")
print(f"   Encoder: {model.cfg.encoder._target_}")
print(f"   Decoder: {model.cfg.decoder._target_}")
print(f"   Vocab size: {model.cfg.decoder.vocab_size}")

In [ ]:
# Setup training data
print(f"⚙️  Setting up training data...")
model.setup_training_data(train_data_config=OmegaConf.create({
    "manifest_filepath": f"{DATASET_DIR}/manifest_train.json",
    "sample_rate": 16000,
    "batch_size": BATCH_SIZE,
    "num_workers": 4,
    "shuffle": True,
    "pin_memory": True,
}))

model.setup_validation_data(val_data_config=OmegaConf.create({
    "manifest_filepath": f"{DATASET_DIR}/manifest_dev.json",
    "sample_rate": 16000,
    "batch_size": BATCH_SIZE,
    "num_workers": 4,
}))

# Update learning rate
model.cfg.optim.lr = LEARNING_RATE
model.cfg.trainer.accumulate_grad_batches = ACCUMULATE_GRAD_BATCHES

print(f"✅ Training config ready")
print(f"   Train manifest: {DATASET_DIR}/manifest_train.json")
print(f"   Val manifest: {DATASET_DIR}/manifest_dev.json")
print(f"   Checkpoint dir: {CHECKPOINT_DIR}")

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

# Setup trainer
trainer = pl.Trainer(
    devices=1,
    accelerator="gpu",
    max_epochs=NUM_EPOCHS,
    num_sanity_val_steps=0,
    enable_checkpointing=True,
    logger=False,
    callbacks=[
        ModelCheckpoint(
            dirpath=CHECKPOINT_DIR,
            filename="{epoch:02d}-{val_loss:.2f}",
            save_last=True,
            monitor="val_loss",
            mode="min",
        ),
        EarlyStopping(
            monitor="val_loss",
            patience=2,
            verbose=True,
        )
    ],
)

print("🚀 Starting fine-tuning...\n")
trainer.fit(model)

print("\n✅ Fine-tuning complete!")

## Save Results

In [ ]:
import glob

# Find best checkpoint
checkpoints = glob.glob(f"{CHECKPOINT_DIR}/*.ckpt")
if checkpoints:
    best_checkpoint = sorted(checkpoints)[-1]
    print(f"📦 Loading best checkpoint: {best_checkpoint}")
    model = models.EncDecRNNTBPEModel.load_from_checkpoint(best_checkpoint)
    print("✅ Model loaded")
else:
    print("⚠️ No checkpoints found")

In [ ]:
from google.colab import files

model_save_path = f"{CHECKPOINT_DIR}/quran_hybrid_finetuned.nemo"
print(f"💾 Saving fine-tuned model to {model_save_path}")
model.save_to(model_save_path)
print("✅ Model saved")

# Create results directory
results_dir = f"{WORK_DIR}/results"
os.makedirs(results_dir, exist_ok=True)
!cp {model_save_path} {results_dir}/
!cp {DATASET_DIR}/tokens.txt {results_dir}/ 2>/dev/null || echo "Tokens copied"

print("\n📥 Files ready for download:")
!ls -lh {results_dir}

print("\n📥 Starting download...")
files.download(results_dir)

In [ ]:
print("="*60)
print("🎉 FINE-TUNING COMPLETE!")
print("="*60)
print()
print("📊 RESULTS SUMMARY:")
print(f"  ✅ Fine-tuned model: quran_hybrid_finetuned.nemo")
print(f"  ✅ Tokenizer: tokens.txt")
print()
print("🔄 NEXT STEPS:")
print("  1. Download results from Colab")
print("  2. Integrate fine-tuned model into your backend")
print("  3. Test on your validation set")
print("  4. Measure PER improvement")
print()
print("✨ You can now use the fine-tuned model for inference!")